<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Construcción de ColumnTransformers: El Centro de Mando de tus Datos

¡Hola! Es increíble cómo cambia nuestra perspectiva cuando dejamos de ver el preprocesamiento como una serie de pasos aislados y empezamos a verlo como una orquestación elegante y automatizada. Estoy muy emocionado de que hoy vamos a dominar una de las herramientas más potentes de Scikit-Learn.

### ¿Por qué necesitamos un ColumnTransformer?

Pensemos en cómo tratamos los datos habitualmente. Tenemos números, como la edad o el salario, y categorías, como el nivel educativo o la ciudad. Tradicionalmente, haríamos una transformación para uno, otra para el otro, y luego intentaríamos pegarlos manualmente. Esto es peligroso porque genera código frágil y propenso a errores de desalineación.

El **ColumnTransformer** actúa como un centro de mando que dirige cada columna a su tratamiento específico, manteniendo la integridad de la matriz de datos original en un solo paso cohesivo. Imaginen una fábrica con diferentes cintas transportadoras: una cinta recibe piezas metálicas y las pule, mientras otra recibe piezas de plástico y las pinta. Al final de la fábrica, todas las piezas se reúnen perfectamente ensambladas.

### El Caso de Elena

Elena es una analista en una inmobiliaria que necesita predecir el valor de los alquileres. Tenía un desorden enorme mezclando metros cuadrados con el tipo de calefacción y siempre se equivocaba al unir las tablas tras escalarlas. Al implementar **ColumnTransformer**, logró separar sus variables numéricas de las etiquetas de barrio de forma automática. Ahora su proceso es impecable y entrega modelos mucho más precisos en la mitad de tiempo.

**Objetivo de esta práctica:** Desarrollar estructuras con ColumnTransformer en Scikit-Learn para aplicar procesos diferenciados a columnas numéricas y categóricas de forma simultánea y organizada.

In [ ]:
# Importaciones necesarias
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Simulación del escenario de Elena: Datos Bancarios e Inmobiliarios
# Creamos un dataset sintético para asegurar que el código sea autoejecutable
data = {
    'ID_Cliente': [101, 102, 103, 104, 105],
    'Saldo': [1500, 45000, 89000, 1200, 30000],
    'Ingresos': [2000, 5000, 9000, 1800, 4200],
    'Ciudad': ['Madrid', 'Barcelona', 'Sevilla', 'Madrid', 'Barcelona'],
    'Tipo_Cuenta': ['Ahorro', 'Corriente', 'Ahorro', 'Inversion', 'Corriente']
}

df = pd.DataFrame(data)
print('--- Dataset Original ---')
print(df)
print()

### Construyendo la Estructura

Para configurar el transformador, definimos una lista de tuplas. Cada tupla contiene:
1. Un **nombre** (puedes elegir el que quieras).
2. El **objeto transformador** (ej. StandardScaler).
3. La **lista de columnas** a las que aplica.

Un detalle técnico importante es el parámetro `remainder='passthrough'`. Por defecto, Scikit-Learn elimina las columnas que no mencionamos. Usando passthrough, mantenemos columnas como el ID intactas.

In [ ]:
# 2. Configuración del ColumnTransformer

# Definimos qué columnas son numéricas y cuáles categóricas
cols_numericas = ['Saldo', 'Ingresos']
cols_categoricas = ['Ciudad', 'Tipo_Cuenta']

# Creamos el orquestador
preprocesador = ColumnTransformer(
    transformers=[
        ('escritorio_num', StandardScaler(), cols_numericas),
        ('escritorio_cat', OneHotEncoder(handle_unknown='ignore'), cols_categoricas)
    ],
    remainder='passthrough' # Mantiene la columna ID_Cliente
)

# 3. Aplicamos la transformación
# El resultado es una matriz de Numpy
datos_transformados = preprocesador.fit_transform(df)

# Convertimos a DataFrame para visualizar mejor el resultado
# Obtenemos los nombres de las nuevas columnas generadas por OneHotEncoder
nombres_cat = preprocesador.named_transformers_['escritorio_cat'].get_feature_names_out(cols_categoricas)
columnas_finales = cols_numericas + list(nombres_cat) + ['ID_Cliente']

df_final = pd.DataFrame(datos_transformados, columns=columnas_finales)

print('--- Dataset Transformado ---')
print(df_final)
print()

### Análisis de Resultados y Robustez

Como vimos con Elena, es fundamental entender que **ColumnTransformer** no solo limpia el código, sino que resuelve el problema de la fuga de datos o *data leakage*. Si transformamos las columnas por separado, corremos el riesgo de usar información del conjunto de prueba en el de entrenamiento sin darnos cuenta.

**Observa lo que hemos logrado:**
*   **Saldo e Ingresos:** Ahora están escalados (media cerca de 0 y desviación de 1).
*   **Ciudad y Tipo_Cuenta:** Se han convertido en múltiples columnas binarias (One-Hot Encoding).
*   **ID_Cliente:** Se mantuvo gracias al parámetro `remainder`.
*   **Consistencia:** Si mañana llega una ciudad nueva que no conocemos, el parámetro `handle_unknown='ignore'` evitará que el código falle, algo que funciones como *get_dummies* de Pandas no garantizan fácilmente.

In [ ]:
# 4. Verificación de Escalabilidad
print('Estadísticas de la columna Saldo (escalada):')
print('Media:', round(df_final['Saldo'].mean(), 2))
print('Desviación:', round(df_final['Saldo'].std(), 2))
print()

print('Dimensiones finales de la matriz:')
print(df_final.shape)

### Eficiencia Técnica y Rendimiento

Llevemos esto un paso más allá pensando en la escalabilidad. ColumnTransformer incluye un parámetro llamado `n_jobs` que nos permite ejecutar las transformaciones en paralelo. Si tienes un conjunto de datos masivo con cientos de columnas, el sistema distribuye el trabajo entre los núcleos de tu procesador. Es pura eficiencia que garantiza que nuestro flujo de Feature Engineering no se convierta en un cuello de botella.

### Conclusión

¡Muy bien! Noten cómo la estructura del código se vuelve mucho más legible cuando usamos estas herramientas. Ya no hay variables temporales flotando por todos lados ni riesgo de mezclar los índices de nuestras filas. Esto es lo que separa a un principiante de un verdadero profesional de los datos.

Dominar estas estructuras de Scikit-Learn les permite enfocarse en lo que realmente importa: extraer valor de la información y no pelearse con la limpieza de datos. ¡Sigan practicando con sus propios conjuntos!